# 16 — Dataset Assembly

Merges all training sources into one clean, balanced dataset ready for fine-tuning.

**Why merge?** The generation notebooks (00–14) accumulate high-quality,
curated pairs in `knowledge_pairs.jsonl`. `10_synthetic_data_generation`
produces raw synthetic samples that `15_validation` validates. Without this
merge step, the final fine-tune (`17_finetune`) would see only the synthetic
data — and all the curated pairs would go to waste.

**What this notebook does:**
1. Load validated synthetic samples from `15_validation` (`04_validated/valid_samples.jsonl`)
2. Load curated knowledge pairs from the generation notebooks (`02_knowledge/knowledge_pairs.jsonl`)
3. Convert everything to ChatML format with the canonical system prompt
4. Deduplicate by instruction prefix
5. Cap each task type using soft type-weighted caps (minority categories uncapped)
6. Shuffle and split into train / valid / test (80 / 10 / 10)
7. Write `mlx/` format ready for `mlx_lm lora --use-chat-template`

**Inputs:**
- `../data/04_validated/valid_samples.jsonl` — validated synthetic samples (from `15_validation`)
- `../data/02_knowledge/knowledge_pairs.jsonl` — curated pairs (from the generation notebooks)

**Output:**
- `../data/05_dataset/train.jsonl` — training samples (full format with metadata)
- `../data/05_dataset/valid.jsonl` — validation samples
- `../data/05_dataset/test.jsonl`  — test samples
- `../data/05_dataset/holdout.jsonl` — persistent held-out eval set (reserved before training, leakage-checked; ARO issue #405)
- `../data/05_dataset/mlx/train.jsonl` — messages-only ChatML, for `mlx_lm lora`
- `../data/05_dataset/mlx/valid.jsonl`
- `../data/05_dataset/stats.json`   — counts, task distribution, active TYPE_CAPS, metadata
- `../data/05_dataset/dataset_report.md` — end-to-end funnel report with per-stage drop accounting
- `../data/05_dataset/drop_reasons.csv`  — per-stage drop reasons for post-hoc analysis
- `run/<date>/16_dataset_assembly.csv` — per-sample metadata for analysis

In [ ]:
import json
import random
from pathlib import Path
from collections import Counter

try:
    SCRIPT_DIR = Path(__file__).parent.resolve()
except NameError:
    SCRIPT_DIR = Path('.').resolve()

import sys, importlib
_cfg_dir = SCRIPT_DIR
if str(_cfg_dir) not in sys.path:
    sys.path.insert(0, str(_cfg_dir))
import config; importlib.reload(config); from config import *

# End-to-end funnel accounting (issue #409): every filter stage below
# records before/after counts and drop reasons; dataset_report.md is
# generated at the end of this notebook.
funnel = FunnelCounter('dataset_assembly')

DATA_IN  = DATA_ROOT / '04_validated'
DATA_OUT = DATA_ROOT / '05_dataset'
MLX_DIR  = DATA_OUT / 'mlx'
DATA_OUT.mkdir(parents=True, exist_ok=True)
MLX_DIR.mkdir(parents=True, exist_ok=True)

# ── Load validated synthetic samples (notebook 09 output) ────────────────────
samples = []
valid_samples_file = DATA_IN / 'valid_samples.jsonl'
if not valid_samples_file.exists():
    raise FileNotFoundError(
        f'{valid_samples_file} not found — run 10_synthetic_data_generation '
        f'and 15_validation first.'
    )
with open(valid_samples_file) as f:
    for line in f:
        if line.strip():
            samples.append(json.loads(line))

print(f'Loaded {len(samples)} validated synthetic samples')
print('Distribution:', dict(Counter(s['task_type'] for s in samples)))
funnel.record_stage('load_validated_synthetic', len(samples), len(samples))

In [ ]:
# ── Load curated knowledge pairs (notebooks 03+05+06+07+08 output) ──────────
# These pairs are already high-quality (exec-validated or doc-grounded).
# They are not in the synthetic samples file, so they would be dropped without
# this merge step — meaning all the effort of NB06/06/07/08 would be wasted.

def _source_to_task_type(source):
    """Map knowledge_pairs source tags to task_type used in this notebook."""
    src = source.lower()
    if any(src.startswith(p) for p in ('book_qa:', 'wiki:', 'actions_explain', 'actions_which')):
        return 'syntax_qa'
    if src.startswith('repair'):
        return 'debugging'
    # example:*, mutation, recombination, spec_to_code, readme_to_code,
    # actions_usage, actions_alias, actions_context, book, aro_by_example, proposal:*
    return 'code_generation'


kp_added = 0
kp_skipped_empty = 0
_n_synthetic = len(samples)
if PAIRS_FILE.exists():
    with open(PAIRS_FILE) as f:
        for line in f:
            if not line.strip():
                continue
            p = json.loads(line)
            if is_jsonl_metadata_record(p):
                continue  # flagged header line (issue #382), not a pair
            instruction = p.get('instruction', '').strip()
            output      = p.get('output', '').strip()
            if not instruction or not output:
                kp_skipped_empty += 1
                continue
            source    = p.get('source', 'knowledge_pairs')
            task_type = _source_to_task_type(source)
            samples.append({
                'task_type':  task_type,
                'instruction': instruction,
                'output':      output,
                'source':      source,
                'score':       p.get('score', 1.0),
                'valid':       True,   # already curated
                'notebook':    p.get('notebook', ''),
                # auto-wrap flag survives to assembly so the synthetic-wrapper
                # share of code_generation can be capped (issue #380)
                'auto_wrapped': bool((p.get('metadata') or {}).get('auto_wrapped')),
            })
            kp_added += 1
else:
    print(f'WARNING: {PAIRS_FILE} not found — run the generation notebooks '
          f'(00_init through 14_comment_extraction) first.')

funnel.record_stage(
    'merge_curated_pairs', _n_synthetic, len(samples),
    reasons=({'curated_empty_instruction_or_output': kp_skipped_empty}
             if kp_skipped_empty else None),
)
print(f'Added {kp_added} curated knowledge pairs '
      f'(skipped {kp_skipped_empty} with empty instruction/output)')
print(f'Total samples before dedup: {len(samples)}')
print('Full distribution:', dict(Counter(s['task_type'] for s in samples)))

## System prompt

In [ ]:
# Load knowledge base and build the canonical system prompt.
# Uses build_system_prompt(kb) from config so NB10 uses the same prompt
# as NB10 and the warm-start NB05 (consistent system token across all training).
with open(KB_FILE) as _kb_f:
    _kb = json.load(_kb_f)

ARO_SYSTEM_PROMPT = build_system_prompt(_kb)
print(f'System prompt: {len(ARO_SYSTEM_PROMPT)} chars  ({len(_kb["actions"])} actions referenced)')

## Convert to ChatML

In [ ]:
def build_messages(sample):
    task  = sample.get('task_type', '')
    instr = sample.get('instruction', '').strip()
    inp   = sample.get('input', '').strip()
    out   = sample.get('output', '').strip()

    # Tool calling: already in multi-turn messages format
    if task == 'tool_calling':
        msgs = sample.get('messages', [])
        if msgs:
            return msgs  # Return full messages list
        return None

    if task == 'fim':
        prefix = sample.get('prefix', '')
        suffix = sample.get('suffix', '')
        middle = sample.get('middle', '')
        if middle.strip():
            user = (
                f'Complete the missing ARO statement(s). Only output the missing line(s).\n\n'
                f'```aro\n{prefix}\n<FILL_HERE>\n{suffix}\n```'
            )
            return user, middle
        elif instr:
            user = instr
            return user, out
        return None

    if not out:
        return None

    if inp:
        user = f'{instr}\n\n```aro\n{inp}\n```'
    else:
        user = instr

    return (user, out) if user.strip() else None


def to_chatml(sample):
    result = build_messages(sample)
    if result is None:
        return None

    # Multi-turn (tool calling): result is a list of messages
    if isinstance(result, list):
        msgs = result
        # Ensure system prompt is present and up-to-date
        if msgs and msgs[0].get('role') == 'system':
            msgs[0]['content'] = ARO_SYSTEM_PROMPT
        else:
            msgs.insert(0, {'role': 'system', 'content': ARO_SYSTEM_PROMPT})
        return {
            'messages': msgs,
            'task_type': sample.get('task_type', ''),
            'source':    sample.get('source', ''),
        }

    # Single-turn (code gen, Q&A, etc.): result is (user, assistant) tuple
    user_content, assistant_content = result
    if len(user_content) < 10 or len(assistant_content) < 10:
        return None
    return {
        'messages': [
            {'role': 'system',    'content': ARO_SYSTEM_PROMPT},
            {'role': 'user',      'content': user_content},
            {'role': 'assistant', 'content': assistant_content},
        ],
        'task_type': sample.get('task_type', ''),
        'source':    sample.get('source', sample.get('domain', sample.get('scenario', ''))),
    }


chatml = []
_chatml_drops = Counter()
for s in samples:
    c = to_chatml(s)
    if c is None:
        task = s.get('task_type', '') or 'unknown'
        if task == 'tool_calling':
            _chatml_drops['tool_calling_missing_messages'] += 1
        elif not (s.get('output') or '').strip():
            _chatml_drops[f'missing_output:{task}'] += 1
        else:
            _chatml_drops[f'too_short_or_invalid:{task}'] += 1
        continue
    chatml.append(c)
funnel.record_stage('chatml_conversion', len(samples), len(chatml), reasons=_chatml_drops)
print(f'Converted: {len(chatml)}  (dropped {len(samples) - len(chatml)} empty)')
drop_rate = (len(samples) - len(chatml)) / max(1, len(samples))
if drop_rate > 0.15:
    print(f'⚠️  HIGH DROP RATE: {drop_rate:.0%} of samples lost during ChatML conversion.')
    print('   Check build_messages() for task types with missing fields.')

## Deduplicate

In [ ]:
seen, deduped = set(), []
for s in chatml:
    key = s['messages'][1]['content'][:300]
    if key not in seen:
        seen.add(key)
        deduped.append(s)
funnel.record_stage('dedup_instruction_prefix', len(chatml), len(deduped),
                    reasons={'duplicate_instruction_prefix': len(chatml) - len(deduped)})
print(f'After dedup: {len(deduped)}')

## Semantic near-duplicate removal (issue #404)

Exact dedup above only catches identical `instruction[:300]` prefixes.
Near-duplicates ("Show me how to use Extract" vs "How do I use Extract?") and
cross-notebook duplicates (NB06 and NB10 generating similar pairs) survive it
and dilute the training signal. Cluster the remaining instructions **per task
type** — all-MiniLM embeddings when `sentence-transformers` is available,
token-overlap Jaccard otherwise — and keep the highest-scored representative
per cluster.

In [ ]:
# ── Semantic near-dedup across all sources (issue #404) ──────────────────────
from collections import defaultdict

def _src_prefix(s):
    return (s.get('source') or 'unknown').split(':')[0]

_before_by_source = Counter(_src_prefix(s) for s in deduped)
_before_total = len(deduped)

_by_task = defaultdict(list)
for s in deduped:
    _by_task[s.get('task_type', 'unknown')].append(s)

_nd_kept = []
_nd_total_dropped = 0
for task, group in sorted(_by_task.items()):
    kept, dropped = near_duplicate_filter(
        group,
        get_text=lambda s: s['messages'][1]['content'] if len(s.get('messages', [])) > 1 else '',
        get_score=lambda s: s.get('score', 1.0),
        label=task,
    )
    _nd_kept.extend(kept)
    _nd_total_dropped += dropped

deduped = _nd_kept
_after_by_source = Counter(_src_prefix(s) for s in deduped)

print(f'\nAfter semantic near-dedup: {_before_total} → {len(deduped)} '
      f'(dropped {_nd_total_dropped})')
print(f'{"source":<28} {"before":>7} {"after":>7}')
for src in sorted(_before_by_source, key=lambda s: -_before_by_source[s])[:15]:
    print(f'{src:<28} {_before_by_source[src]:>7} {_after_by_source.get(src, 0):>7}')

## Soft type-weighted caps (minority categories uncapped)

In [ ]:
# TYPE_CAPS moved to config.py (issue #406) — versioned, with a changelog.
# The active caps and TYPE_CAPS_VERSION are recorded in stats.json below so
# every dataset can be attributed to the exact caps that shaped it.
print(f'Using TYPE_CAPS {TYPE_CAPS_VERSION} from config.py:')
for _t, _cap in TYPE_CAPS.items():
    print(f'  {_t:25s}: {_cap if _cap is not None else "uncapped"}')
print()

capped, type_counts = [], Counter()
_cap_drops = Counter()
for s in deduped:
    t = s.get('task_type', 'unknown')
    cap = TYPE_CAPS.get(t, DEFAULT_TYPE_CAP)
    if cap is None or type_counts[t] < cap:
        capped.append(s)
        type_counts[t] += 1
    else:
        _cap_drops[f'cap_exceeded:{t}'] += 1

total_capped = len(capped)
print(f'After type-weighted cap: {total_capped}')
for t, n in sorted(type_counts.items(), key=lambda x: -x[1]):
    pct = 100 * n / max(1, total_capped)
    flag = '  ⚠️  DOMINANT' if pct > 45 else ''
    print(f'  {t:25s}: {n:5d}  ({pct:.1f}%){flag}')

funnel.record_stage(f'type_caps[{TYPE_CAPS_VERSION}]', len(deduped), len(capped),
                    reasons=_cap_drops)

# Soft warning at 50% (issue #406), hard error at 65%: a dominant task
# type skews training long before it trips the hard gate.
for t, n in type_counts.items():
    share = n / max(1, total_capped)
    if share > 0.50:
        print(f'⚠️  WARNING: task type "{t}" is {share:.0%} of the dataset (> 50%) — '
              f'consider tightening its cap in config.TYPE_CAPS or adding minority data.')
    if share > 0.65:
        raise ValueError(
            f'Task type "{t}" makes up {share:.0%} of the dataset — training will be skewed. '
            f'Lower the cap for "{t}" or add more data for other task types.'
        )

## Auto-wrap share cap for code_generation (issue #380)

Auto-wrapped pairs are synthetic `(Application-Start: Example) { … }` wrappers
around bare snippets (NB04 Phase 4). Both bare REPL statements and complete
feature sets are valid training targets — but synthetic wrappers must not
drown out real hand-written feature sets, or the model loses its calibration
for when a wrapper is actually required. Cap their share of `code_generation`
at `AUTO_WRAP_MAX_SHARE` (config.py).

In [ ]:
# ── Cap the auto-wrapped share of code_generation pairs (issue #380) ─────────
random.seed(380)

_cg = [s for s in capped if s.get('task_type') == 'code_generation']
_cg_wrapped = [s for s in _cg if s.get('auto_wrapped')]
_n_real = len(_cg) - len(_cg_wrapped)
_share = len(_cg_wrapped) / max(1, len(_cg))
print(f'code_generation: {len(_cg_wrapped)}/{len(_cg)} auto-wrapped '
      f'({_share:.0%}, cap {AUTO_WRAP_MAX_SHARE:.0%})')

if _cg and _share > AUTO_WRAP_MAX_SHARE:
    # w/(w+r) <= cap  →  w <= cap/(1-cap) * r
    _allowed = int(AUTO_WRAP_MAX_SHARE / (1 - AUTO_WRAP_MAX_SHARE) * _n_real)
    _excess = len(_cg_wrapped) - _allowed
    if _excess > 0:
        # Drop the lowest-scored wrapped pairs first (seeded random tiebreak)
        _order = sorted(_cg_wrapped, key=lambda s: (s.get('score', 1.0), random.random()))
        _drop_ids = {id(s) for s in _order[:_excess]}
        capped = [s for s in capped if id(s) not in _drop_ids]
        type_counts = Counter(s.get('task_type', 'unknown') for s in capped)
        print(f'  dropped {_excess} auto-wrapped code_generation pairs '
              f'(kept {_allowed} wrapped / {_n_real} real)')
print(f'After auto-wrap cap: {len(capped)}')

In [ ]:
import re as _re

def _is_plausible(sample):
    msgs = sample.get('messages', [])
    task = sample.get('task_type', '')

    # Tool calling samples are multi-turn with pre-validated format — skip content checks
    if task == 'tool_calling':
        return (len(msgs) >= 3, 'too_few_messages') if len(msgs) < 3 else (True, '')

    user = msgs[1]['content'] if len(msgs) > 1 else sample.get('instruction', '')
    asst = msgs[2]['content'] if len(msgs) > 2 else sample.get('output', '')

    if len(user.strip()) < 20 or len(asst.strip()) < 15:
        return False, 'too_short'
    if _re.match(r'^[!?.,\s]{10,}$', asst.strip()):
        return False, 'garbage_output'

    u_tok = set(user.lower().split())
    a_tok = set(asst.lower().split())
    if a_tok and len(a_tok & u_tok) / len(a_tok) > 0.85:
        return False, 'copy_ratio'

    # Code tasks MUST contain code blocks
    if task in ('code_generation', 'debugging', 'fim'):
        if '```' not in asst:
            return False, 'missing_code_block'

    # Q&A: must have explanatory text (not just code)
    if task == 'syntax_qa':
        text_only = _re.sub(r'```.*?```', '', asst, flags=_re.DOTALL).strip()
        if len(text_only) < 15:
            return False, 'qa_no_explanation'

    return True, ''

plausible = []
rejected_plausibility = Counter()
for s in capped:
    ok, reason = _is_plausible(s)
    if ok:
        plausible.append(s)
    else:
        rejected_plausibility[reason] += 1

funnel.record_stage('plausibility_filter', len(capped), len(plausible),
                    reasons=rejected_plausibility)
print(f'Plausibility filter: {len(capped)} → {len(plausible)} ({len(capped)-len(plausible)} rejected)')
for reason, count in rejected_plausibility.most_common():
    print(f'  {reason}: {count}')

final = plausible

## Per-source quality scores + soft down-weighting (issue #407)

A noisy external-repo README pair must not count the same as a hand-curated
`Examples/` pair. Every sample gets a per-source quality `weight` (static
scores from `config.SOURCE_QUALITY_SCORES`, blended 50/50 with pass rates
auto-derived from NB15 validation results when available). Sources above
`SOURCE_SOFT_CAP_SHARE` of the dataset are *softly* down-sampled — the keep
probability interpolates with quality, so high-quality sources keep more of
their excess than noisy ones — instead of being hard-capped. Any source above
`SOURCE_SHARE_FLAG` of the final set is flagged.

In [ ]:
# ── Per-source quality weighting (issue #407) ─────────────────────────────────
# 1. Blend static quality scores with NB15-derived pass rates when available.
_derived_rates = {}
_all_samples_file = DATA_IN / 'all_samples.jsonl'
if _all_samples_file.exists():
    _val_records = []
    with open(_all_samples_file) as f:
        for line in f:
            if line.strip():
                try:
                    _val_records.append(json.loads(line))
                except Exception:
                    pass
    _derived_rates = derive_source_quality_from_validation(_val_records)
    print(f'Derived NB15 pass rates for {len(_derived_rates)} source prefixes')

def _quality(s):
    static = source_quality_score(s.get('source', ''), s.get('notebook'))
    prefix = (s.get('source') or 'unknown').split(':')[0].strip().lower()
    if prefix in _derived_rates:
        return 0.5 * static + 0.5 * _derived_rates[prefix]
    return static

for s in final:
    s['weight'] = round(_quality(s), 3)

# 2. Soft down-weighting for sources above SOURCE_SOFT_CAP_SHARE.
random.seed(407)
_total = len(final)
_src_counts = Counter((s.get('source') or 'unknown').split(':')[0] for s in final)
_soft_kept = []
_soft_dropped = Counter()
for s in final:
    prefix = (s.get('source') or 'unknown').split(':')[0]
    share = _src_counts[prefix] / max(1, _total)
    if share <= SOURCE_SOFT_CAP_SHARE:
        _soft_kept.append(s)
        continue
    # keep probability interpolates between the cap ratio (quality→0) and
    # 1.0 (quality→1.0): high-quality sources keep more of their excess.
    base = SOURCE_SOFT_CAP_SHARE / share
    keep_prob = min(1.0, base + (1.0 - base) * s['weight'])
    if random.random() < keep_prob:
        _soft_kept.append(s)
    else:
        _soft_dropped[prefix] += 1

if _soft_dropped:
    print(f'Soft down-weighting (sources above {SOURCE_SOFT_CAP_SHARE:.0%} share):')
    for prefix, n in _soft_dropped.most_common():
        print(f'  {prefix:<25} dropped {n}')
else:
    print(f'No source exceeds {SOURCE_SOFT_CAP_SHARE:.0%} share — nothing down-weighted.')
final = _soft_kept

# 3. Per-source share report — flag anything above SOURCE_SHARE_FLAG.
_final_counts = Counter((s.get('source') or 'unknown').split(':')[0] for s in final)
print(f'\nPer-source share of final dataset ({len(final)} samples):')
for prefix, n in _final_counts.most_common(20):
    share = n / max(1, len(final))
    flag = f'  ⚠ >{SOURCE_SHARE_FLAG:.0%} — dominating source' if share > SOURCE_SHARE_FLAG else ''
    print(f'  {prefix:<25} {n:>6}  ({share:5.1%})  quality={source_quality_score(prefix):.2f}{flag}')

## Train / valid / test split  (80 / 10 / 10)

In [ ]:
# Split changed from 90/5/5 to 80/10/10 (2026-04-22) — the old 5% test set
# was only 38 samples, too small to measure changes under +-15 ppt.
# Complete-program filter — drop fragment-only code/debug samples.
# Fragments are the main reason students fail to wrap ARO in `(name: activity) { }`.
_before_frag = len(final)
final, _frag_dropped = filter_complete_program_samples(final)
funnel.record_stage('complete_program_filter', _before_frag, len(final),
                    reasons={'fragment_only_code_sample': _frag_dropped})
print(f'Complete-program filter: dropped {_frag_dropped} fragment-only code samples '
      f'(kept {len(final)})')

# ── Held-out evaluation set (issue #405) ────────────────────────────────────
# Reserve a persistent held-out set BEFORE the split. First run: 5% stratified
# by task type, written to holdout.jsonl. Later runs: the same file is
# re-applied, and its samples are verifiably removed from the train pool —
# so NB19/NB20 can measure generalization instead of memorization.
from leakage import reserve_holdout, verify_exclusion

HOLDOUT_FILE = DATA_OUT / 'holdout.jsonl'
final, holdout = reserve_holdout(final, HOLDOUT_FILE,
                                 fraction=0.05, min_size=40, seed=1234)
_collisions = verify_exclusion(final, holdout)
if _collisions:
    raise ValueError(f'{len(_collisions)} train-pool samples still collide '
                     f'with the held-out set — holdout exclusion is broken.')
print(f'Held-out eval set: {len(holdout)} samples at {HOLDOUT_FILE.name} '
      f'(verifiably excluded from train/valid/test)')

random.seed(42)
random.shuffle(final)
n     = len(final)
t_end = int(n * 0.80)
v_end = int(n * 0.90)
train = final[:t_end]
valid = final[t_end:v_end]
test  = final[v_end:]
print(f'train={len(train)}  valid={len(valid)}  test={len(test)}')

## Save full format + mlx format

In [ ]:
from corpus_sanitize import sanitize_assistant_text, gate_check

def _sanitize_record(rec):
    """Strip URLs/HTML/tool-signatures from assistant content in a sample."""
    msgs = rec.get('messages', [])
    for m in msgs:
        if isinstance(m, dict) and m.get('role') == 'assistant' and isinstance(m.get('content'), str):
            m['content'] = sanitize_assistant_text(m['content'])
    if isinstance(rec.get('output'), str):
        rec['output'] = sanitize_assistant_text(rec['output'])
    return rec

# Sanitise every split before writing — last defense against polluted
# samples reaching mlx-lm. Round-3 shipped a model that emitted
# `![](https://via.placeholder.com/...)` and `read_file(path)` because
# the corpus carried both patterns from scraped READMEs and from the
# system prompt baked into every training pair.
train = [_sanitize_record(s) for s in train]
valid = [_sanitize_record(s) for s in valid]
test  = [_sanitize_record(s) for s in test]

# Cleanliness gate — fail loudly if contamination still exceeds thresholds.
def _assistant_text(s):
    msgs = s.get('messages', [])
    for m in msgs:
        if isinstance(m, dict) and m.get('role') == 'assistant':
            return m.get('content') or ''
    return s.get('output') or ''

print('Corpus cleanliness gate (train split):')
gate_check(train, get_text=_assistant_text)
print('Corpus cleanliness gate (valid split):')
gate_check(valid, get_text=_assistant_text)


def save_jsonl(data, path):
    with open(path, 'w') as f:
        for item in data:
            f.write(json.dumps(item) + '\n')
    print(f'  {path.name}: {len(data)} samples')


# Full format (with task_type / source metadata)
save_jsonl(train, DATA_OUT / 'train.jsonl')
save_jsonl(valid, DATA_OUT / 'valid.jsonl')   # NOTE: valid (not val) — mlx-lm convention
save_jsonl(test,  DATA_OUT / 'test.jsonl')

# mlx-lm format: only the "messages" key (mlx_lm.lora --use-chat-template reads this)
save_jsonl([{'messages': s['messages']} for s in train], MLX_DIR / 'train.jsonl')
save_jsonl([{'messages': s['messages']} for s in valid], MLX_DIR / 'valid.jsonl')

# Stats
stats = {
    'total': len(final), 'train': len(train), 'valid': len(valid), 'test': len(test),
    'task_counts': dict(type_counts),
    'avg_user_len':      int(sum(len(s['messages'][1]['content']) for s in final) / max(1, len(final))),
    'avg_assistant_len': int(sum(len(s['messages'][2]['content']) for s in final) / max(1, len(final))),
    # Active caps recorded at assembly time (issue #406) — dataset-shape
    # changes are attributable to data changes vs. cap changes.
    'type_caps_version': TYPE_CAPS_VERSION,
    'type_caps':         TYPE_CAPS,
    # Generation metadata (issue #382)
    '_metadata':         build_artifact_metadata(),
}
with open(DATA_OUT / 'stats.json', 'w') as f:
    json.dump(stats, f, indent=2)

print(f'\nstats.json: avg user={stats["avg_user_len"]} chars, avg assistant={stats["avg_assistant_len"]} chars')


## Leakage Check (issue #405)

Before anyone trusts an eval number, verify the training instructions do not
overlap the evaluation prompts. Two checks per eval set:

- **exact** — normalised 120-char instruction prefix appears verbatim in train
- **near** — character-3-gram Jaccard ≥ 0.85 against any train instruction
  (catches template-level paraphrases without an embedding model)

Checked sets: `valid`, `test`, the held-out set, and `Train/eval_prompts.json`
(the prompts NB19/NB20 and the packaged-model evals draw from). Exact overlap
with the held-out set or eval_prompts fails the build; near-duplicates inside
valid/test are reported only (they share generation templates with train by
construction).

In [ ]:
from leakage import leakage_report, print_leakage_report, sample_instruction

_train_instrs = [sample_instruction(s) for s in train]

_eval_sets = {
    'valid':   [sample_instruction(s) for s in valid],
    'test':    [sample_instruction(s) for s in test],
    'holdout': [sample_instruction(s) for s in holdout],
}
_eval_prompts_file = TRAIN_ROOT / 'eval_prompts.json'
if _eval_prompts_file.exists():
    with open(_eval_prompts_file) as _f:
        _eval_sets['eval_prompts'] = [p['prompt'] for p in json.load(_f)
                                      if p.get('prompt')]

print(f'Leakage check: {len(_train_instrs)} train instructions vs '
      f'{sum(len(v) for v in _eval_sets.values())} eval prompts ...')
leak = leakage_report(_train_instrs, _eval_sets)
print_leakage_report(leak)

from datetime import date as _date
_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)
with open(_run_dir / '16_leakage_report.json', 'w') as _f:
    json.dump(leak, _f, indent=2)
print(f"Saved: {_run_dir / '16_leakage_report.json'}")

# Hard gate: the truly-held-out sets must have ZERO exact overlap with train.
for _name in ('holdout', 'eval_prompts'):
    _r = leak.get(_name)
    if _r and _r['exact'] > 0:
        raise ValueError(
            f'{_r["exact"]} {_name} prompts appear VERBATIM in the training '
            f'set — eval metrics would measure memorization. Fix the '
            f'holdout exclusion / prompt sources before training.')
# valid/test share templates with train; exact overlap still means the dedup
# upstream failed.
for _name in ('valid', 'test'):
    _r = leak.get(_name)
    if _r and _r['exact'] > 0:
        print(f'WARNING: {_r["exact"]} {_name} samples share an exact '
              f'instruction prefix with train — dedup upstream is leaking.')

## Sanity check

In [ ]:
for s in random.sample(train, 3):
    print('─' * 60)
    print(f'Task: {s["task_type"]}')
    print(f'User:      {s["messages"][1]["content"][:120]}...')
    print(f'Assistant: {s["messages"][2]["content"][:120]}...')
print('─' * 60)

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.style.use('default')
plt.rcParams.update({
    'text.color': '#222222',
    'axes.labelcolor': '#222222',
    'xtick.color': '#333333',
    'ytick.color': '#333333',
    'axes.titlecolor': '#111111',
    'legend.edgecolor': '#cccccc',
    'legend.facecolor': 'white',
    'legend.framealpha': 1.0,
    'figure.facecolor': '#fafafa',
    'axes.facecolor': '#f9f9f9',
    'savefig.facecolor': '#fafafa',
    'savefig.bbox': 'tight',
    'savefig.dpi': 150,
})
from pathlib import Path
from datetime import date as _date

_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)
_out = _run_dir / '16_dataset_assembly.png'

_task_counts = dict(type_counts)
_labels = list(_task_counts.keys())
_sizes  = list(_task_counts.values())
_colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6', '#1abc9c', '#e67e22'][:len(_labels)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: donut — task type composition ──────────────────────────────────────
wedges, _, autotexts = ax1.pie(
    _sizes, labels=_labels, colors=_colors, autopct='%1.0f%%',
    startangle=90, explode=[0.03] * len(_labels),
    pctdistance=0.78, wedgeprops={'edgecolor': 'white', 'linewidth': 2},
)
for t in autotexts:
    t.set_fontsize(9); t.set_fontweight('bold')
ax1.add_artist(plt.Circle((0, 0), 0.50, fc='white'))
ax1.text(0, 0, f'{sum(_sizes):,}\nsamples', ha='center', va='center',
         fontsize=12, fontweight='bold', color='#2c3e50')
ax1.set_title('Task Type Composition', fontweight='bold', pad=15)

# ── Right: bar — train / valid / test split ───────────────────────────────────
_split_labels  = ['Train', 'Valid', 'Test']
_split_values  = [len(train), len(valid), len(test)]
_split_colors  = ['#3498db', '#f39c12', '#2ecc71']
bars = ax2.bar(_split_labels, _split_values, color=_split_colors, edgecolor='white', width=0.5)
ax2.set_ylabel('Samples')
ax2.set_title('Train / Valid / Test Split', fontweight='bold')
ax2.grid(axis='y', alpha=0.3)
for bar, n in zip(bars, _split_values):
    ax2.text(bar.get_x() + bar.get_width() / 2, n + max(_split_values) * 0.01,
             f'{n:,}', ha='center', va='bottom', fontsize=11, fontweight='bold')

fig.suptitle(
    f'Dataset Assembly — {sum(_sizes):,} total  ·  '
    f'avg {stats["avg_user_len"]} chars user  /  {stats["avg_assistant_len"]} chars assistant',
    fontsize=13, fontweight='bold', y=1.01
)
fig.tight_layout()
fig.savefig(_out)
plt.close(fig)
print(f'Saved: {_out}')


In [ ]:
# ── CSV export for downstream analysis ────────────────────────────────────────
import csv
from datetime import date as _date

_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)
csv_path = _run_dir / '16_dataset_assembly.csv'

def _has_code(text):
    return '```' in text

_all_labelled = (
    [(s, 'train') for s in train] +
    [(s, 'valid') for s in valid] +
    [(s, 'test')  for s in test]
)

with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['task_type', 'source_notebook', 'split',
                     'user_length', 'assistant_length', 'has_code'])
    for s, split in _all_labelled:
        msgs = s.get('messages', [])
        user_text = msgs[1]['content'] if len(msgs) > 1 else ''
        asst_text = msgs[2]['content'] if len(msgs) > 2 else ''
        writer.writerow([
            s.get('task_type', ''),
            s.get('source', ''),
            split,
            len(user_text),
            len(asst_text),
            _has_code(asst_text),
        ])

print(f'Exported {len(_all_labelled)} rows to {csv_path}')

## Summary

In [ ]:
print('=' * 65)
print('notebook 16 — DATASET ASSEMBLY SUMMARY')
print('=' * 65)

_tc = dict(type_counts)
_total = sum(_tc.values())

print(f'\nInput sources:')
print(f'  Synthetic samples (NB10/13):  {len([s for s in samples if s.get("valid") is not False and "domain" in s]):>6,}')
print(f'  Curated pairs (knowledge_pairs.jsonl): {kp_added:>6,}')

print(f'\nPipeline funnel:')
print(f'  Raw combined:       {len(samples):>6,}')
print(f'  After ChatML conv:  {len(chatml):>6,}  (dropped {len(samples)-len(chatml)} empty/invalid)')
print(f'  After dedup:        {len(deduped):>6,}')
print(f'  After balance cap:  {_total:>6,}  (type-weighted caps)')
print(f'  After plausibility: {len(final):>6,}  (rejected {_total - len(final)} implausible)')
print(f'  After quality filt: {len(train)+len(valid)+len(test):>6,}')

print(f'\nTask type distribution (after cap):')
for t, n in sorted(_tc.items(), key=lambda x: -x[1]):
    bar = '█' * int(30 * n / max(1, _total))
    pct = 100 * n / max(1, _total)
    flag = '  ← majority' if pct > 40 else ''
    print(f'  {t:25s} {n:5d}  {pct:5.1f}%  {bar}{flag}')

print(f'\nFinal split:')
print(f'  Train: {len(train):,}  ({100*len(train)/max(1,len(train)+len(valid)+len(test)):.0f}%)')
print(f'  Valid: {len(valid):,}  ({100*len(valid)/max(1,len(train)+len(valid)+len(test)):.0f}%)')
print(f'  Test:  {len(test):,}  ({100*len(test)/max(1,len(train)+len(valid)+len(test)):.0f}%)')

print(f'\nAvg lengths (train):')
_u_lens = [len(s['messages'][1]['content']) for s in train]
_a_lens = [len(s['messages'][2]['content']) for s in train]
print(f'  User prompt:      {sum(_u_lens)//max(1,len(_u_lens)):,} chars  '
      f'  (min {min(_u_lens)}, max {max(_u_lens)})')
print(f'  Assistant answer: {sum(_a_lens)//max(1,len(_a_lens)):,} chars  '
      f'  (min {min(_a_lens)}, max {max(_a_lens)})')

print(f'\nOutput files:')
print(f'  {DATA_OUT / "train.jsonl"}')
print(f'  {DATA_OUT / "valid.jsonl"}')
print(f'  {MLX_DIR  / "train.jsonl"}  ← used by 17_finetune')
print(f'  {DATA_OUT / "stats.json"}')
print('=' * 65)

## Data Quality Checks

In [ ]:
# ── Quality filters applied to training set ───────────────────────────────────
# Remove samples that are likely to hurt training:
#   1. Too short (< 30 chars user or < 20 chars assistant) — no signal
#   2. Too repetitive — assistant is just a copy of the user prompt (ratio > 80%)
#   3. Obvious junk — assistant is all punctuation / whitespace / numbers only
#   4. Placeholder outputs — literally "..." or "None" or empty code blocks

import re as _re

_MIN_USER_LEN   = 30
_MIN_ASST_LEN   = 20
_MAX_COPY_RATIO = 0.80   # flag if assistant overlaps > 80% of user tokens

def _token_set(text):
    return set(_re.findall(r'\w+', text.lower()))

def _is_junk_output(text):
    stripped = text.strip()
    if stripped in ('...', 'None', 'null', ''):
        return True
    # Pure punctuation / whitespace
    if _re.fullmatch(r'[^a-zA-Z0-9\n]+', stripped):
        return True
    # Exclamation-mark collapse (the exact symptom we observed post-training)
    if _re.fullmatch(r'!+', stripped):
        return True
    # Empty code fence
    if stripped in ('```aro\n```', '```\n```', '```aro```'):
        return True
    return False

def _is_copy(user, asst):
    u_tok = _token_set(user)
    a_tok = _token_set(asst)
    if not a_tok:
        return True
    overlap = len(u_tok & a_tok) / len(a_tok)
    return overlap > _MAX_COPY_RATIO

quality_issues = []
clean_train = []
for s in train:
    user = s['messages'][1]['content']
    asst = s['messages'][2]['content']

    if len(user) < _MIN_USER_LEN:
        quality_issues.append(('too_short_user',  user[:80]))
        continue
    if len(asst) < _MIN_ASST_LEN:
        quality_issues.append(('too_short_asst',  asst[:80]))
        continue
    if _is_junk_output(asst):
        quality_issues.append(('junk_output',     asst[:80]))
        continue
    if _is_copy(user, asst):
        quality_issues.append(('copy_of_prompt',  asst[:80]))
        continue
    clean_train.append(s)

removed = len(train) - len(clean_train)
funnel.record_stage('quality_filter_train', len(train), len(clean_train),
                    reasons=Counter(k for k, _ in quality_issues))
print(f'Quality filter: removed {removed} samples from train set')
print(f'  Remaining train: {len(clean_train)}')
if quality_issues:
    issue_counts = Counter(k for k, _ in quality_issues)
    for issue, cnt in issue_counts.most_common():
        print(f'  {issue}: {cnt}')
    if removed > 0:
        print('\nExample removed samples:')
        for issue, snippet in quality_issues[:3]:
            print(f'  [{issue}] {repr(snippet[:60])}')

# Guard: fail if more than 20% of the training set was junk
if removed / max(1, len(train)) > 0.20:
    raise ValueError(
        f'{removed}/{len(train)} train samples ({removed/len(train):.0%}) failed quality checks. '
        f'Investigate the data pipeline — too much junk to proceed safely.'
    )

# Replace train with the clean version
train = clean_train
print(f'\n✓ Quality check passed — {len(train)} clean training samples')

# Rewrite mlx train.jsonl with clean samples
def save_jsonl(data, path):
    with open(path, 'w') as f:
        for item in data:
            f.write(json.dumps(item) + '\n')

save_jsonl([{'messages': s['messages']} for s in train], MLX_DIR / 'train.jsonl')
print(f'Rewrote: {MLX_DIR / "train.jsonl"}')

## Dataset funnel report

End-to-end retention funnel with per-stage drop accounting (issue #409).
Writes `dataset_report.md` and `drop_reasons.csv` next to the dataset so
skewed distributions and systematic rejection biases become visible *before*
an expensive training run — not after a bad eval.

In [ ]:
# ── Dataset funnel report (issue #409) ───────────────────────────────────────
import re as _re
from collections import Counter as _Counter

final_all = train + valid + test   # train is already quality-filtered

# Counts per source × task type
_src_task = _Counter()
for s in final_all:
    src = (s.get('source') or 'unknown').split(':')[0] or 'unknown'
    _src_task[(src, s.get('task_type', 'unknown'))] += 1

def _percentiles(values, ps=(5, 25, 50, 75, 95)):
    if not values:
        return {p: 0 for p in ps}
    vs = sorted(values)
    return {p: vs[min(len(vs) - 1, int(len(vs) * p / 100))] for p in ps}

# Instruction-length and code-size distributions
_instr_lens = [len(s['messages'][1]['content'])
               for s in final_all if len(s.get('messages', [])) > 2]

_code_block_re = _re.compile(r'```(?:aro)?\n(.*?)```', _re.DOTALL)
_code_sizes = []
for s in final_all:
    msgs = s.get('messages', [])
    asst = msgs[-1].get('content', '') if msgs and msgs[-1].get('role') == 'assistant' else ''
    blocks = _code_block_re.findall(asst)
    if blocks:
        _code_sizes.append(sum(len(b) for b in blocks))

# Drop-reason CSV for post-hoc analysis
drop_csv_path = DATA_OUT / 'drop_reasons.csv'
funnel.write_drop_csv(drop_csv_path)

_meta = build_artifact_metadata()
lines = ['# Dataset assembly report', '']
lines.append(f"Generated: {_meta['generated_at']}  ·  session `{_meta['session_id']}`  ·  "
             f"ARO-Lang `{(_meta['aro_lang_commit'] or 'unknown')[:12]}`  ·  "
             f"caps `{TYPE_CAPS_VERSION}`")
lines.append('')
lines.append(funnel.render_markdown())
lines.append('')
lines.append('## Final splits')
lines.append('')
lines.append('| Split | Samples |')
lines.append('|-------|--------:|')
for _name, _part in (('train', train), ('valid', valid), ('test', test)):
    lines.append(f'| {_name} | {len(_part):,} |')
lines.append('')
lines.append('## Counts per source × task type')
lines.append('')
lines.append('| Source | Task type | Count |')
lines.append('|--------|-----------|------:|')
for (_src, _task), _n in sorted(_src_task.items(), key=lambda kv: -kv[1]):
    lines.append(f'| {_src} | {_task} | {_n:,} |')
lines.append('')
lines.append('## Length distributions (chars)')
lines.append('')
lines.append('| Percentile | Instruction length | Code size (assistant ```aro``` blocks) |')
lines.append('|-----------:|-------------------:|----------------------------------------:|')
_ip = _percentiles(_instr_lens)
_cp = _percentiles(_code_sizes)
for _p in (5, 25, 50, 75, 95):
    lines.append(f'| p{_p} | {_ip[_p]:,} | {_cp[_p]:,} |')
lines.append('')
lines.append(f'Drop-reason details: `{drop_csv_path.name}`')

report_path = DATA_OUT / 'dataset_report.md'
report_path.write_text('\n'.join(lines) + '\n')

print(f'Report:           {report_path}')
print(f'Drop reasons CSV: {drop_csv_path}')
print()
print(funnel.render_markdown())